# visual-tcav — End-to-End Demo

This notebook demonstrates the full capabilities of the `visual-tcav` package using ResNet50 pretrained on ImageNet.

**What you will see:**
- How to load any PyTorch CNN model
- Multiple ways to instantiate `LocalVisualTCAV` and `GlobalVisualTCAV`
- How to skip explicit `predict()` calls
- How to plug in a custom CAV computation function
- How to interpret concept maps and attribution scores

**Reference paper:**  
De Santis et al., *Visual-TCAV: Concept-based Attribution and Saliency Maps for Post-hoc Explainability in Image Classification*, 2025. [arXiv:2411.05698](https://arxiv.org/abs/2411.05698)

---

## 0. Installation

```bash
pip install visual-tcav
```

For the Text-to-Concept extension (requires CLIP):
```bash
pip install visual-tcav[text-to-concept]
```

## 1. Setup

Import the package and set up paths to concept and test images.

The concept images are organized following the standard `visual-tcav` folder structure:
```
data/
├── concept_images/
│   ├── striped/       ← ~50 images of striped textures (DTD dataset)
│   ├── dotted/        ← ~50 images of dotted textures
│   ├── zigzagged/
│   ├── waffled/
│   ├── honeycombed/
│   ├── chequered/
│   └── random/        ← ~50 random images (negative examples)
└── test_images/
    ├── zebra.jpg
    ├── honeycomb.jpg
    ├── waffle_iron.jpg
    └── zebra/         ← folder of zebra images for GlobalVisualTCAV
```

In [ ]:
import os
import torch
import torchvision.models as models
import matplotlib.pyplot as plt

from visual_tcav import (
    LocalVisualTCAV,
    GlobalVisualTCAV,
    TorchModelWrapper,
    Cav,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Paths — adjust if your data folder is in a different location
DATA_DIR        = "./data"
CONCEPT_DIR     = os.path.join(DATA_DIR, "concept_images")
RANDOM_DIR      = os.path.join(CONCEPT_DIR, "random")
TEST_IMAGES_DIR = os.path.join(DATA_DIR, "test_images")
CACHE_DIR       = os.path.join(DATA_DIR, ".cache")

print(f"\nConcept images: {CONCEPT_DIR}")
print(f"Test images:    {TEST_IMAGES_DIR}")
print(f"Cache:          {CACHE_DIR}")

## 2. Load the model

Load ResNet50 pretrained on ImageNet and wrap it with `TorchModelWrapper`.

The wrapper automatically loads:
- The 1000 ImageNet class labels
- The correct preprocessing (resize, crop, normalization)
- The input size (3, 224, 224)

In [ ]:
# Load ResNet50 with default ImageNet weights
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet._weights = models.ResNet50_Weights.DEFAULT

wrapper = TorchModelWrapper(
    model_name="resnet50",
    model=resnet,
)

print(f"Model loaded: {wrapper.model_name}")
print(f"Number of classes: {len(wrapper.labels)}")
print(f"Input size: {wrapper.input_size}")
print(f"Device: {wrapper.device}")

### Available layers

Use `wrapper.info()` to see which layers can be analyzed.
Deeper layers (closer to the output) capture higher-level concepts like textures and object parts.

In [ ]:
wrapper.info()

---

## 3. LocalVisualTCAV — Style A: everything in the constructor

The most concise way to use `LocalVisualTCAV`.
All parameters are passed directly to the constructor.

This is the recommended style for quick experiments.

In [ ]:
tcav_local = LocalVisualTCAV(
    model_wrapper=wrapper,
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    cache_dir=CACHE_DIR,
)

### 3.1 Predict

`predict()` runs the image through ResNet50 and returns the top predicted classes.
Call `.info()` on the result to print a formatted table.

In [ ]:
predictions = tcav_local.predict()
predictions.info(num_of_classes=5)

### 3.2 Explain

`explain()` runs the full Visual-TCAV pipeline:
1. Computes CAVs for each (concept, layer) pair
2. Extracts feature maps from the test image
3. Computes concept maps (WHERE is the concept?)
4. Computes attribution scores (HOW MUCH does the concept matter?)

Results are cached to `.cache/` to avoid recomputation on subsequent runs.

In [ ]:
tcav_local.explain(cache_cav=True, cache_random=True)

### 3.3 Visualize

`plot()` creates a grid showing:
- **Left columns:** concept map heatmap overlaid on the original image
  (red/yellow = high concept presence, black = absent)
- **Right column:** attribution scores per class as a horizontal bar chart

In [ ]:
tcav_local.plot(figsize=(14, 8))

---

## 4. LocalVisualTCAV — Style B: step-by-step setup

An alternative instantiation style where each setup step is called explicitly.
This gives more control and is easier to debug interactively.

In [ ]:
# Create the explainer with no configuration
tcav_stepbystep = LocalVisualTCAV(
    model_wrapper=wrapper,
    cache_dir=CACHE_DIR,
)

# Set each component separately
tcav_stepbystep.set_test_image(os.path.join(TEST_IMAGES_DIR, "honeycomb.jpg"))
tcav_stepbystep.set_concepts(
    concept_names=["honeycombed", "waffled", "chequered"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
)
tcav_stepbystep.set_layers(["layer3", "layer4"])

### 4.1 Skipping explicit predict()

`explain()` automatically calls `predict()` internally if you have not called it explicitly.
This means you never need to worry about calling methods in the wrong order.

In [ ]:
# No explicit predict() call needed — explain() handles it automatically
tcav_stepbystep.explain(cache_cav=True, cache_random=True)

In [ ]:
tcav_stepbystep.plot(figsize=(18, 10))

---

## 5. LocalVisualTCAV — Multiple layers comparison

Analyzing multiple layers shows how concept detection changes with depth.
Early layers detect simple textures; later layers detect higher-level patterns.

In [ ]:
tcav_multilayer = LocalVisualTCAV(
    model_wrapper=wrapper,
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer2", "layer3", "layer4"],
    cache_dir=CACHE_DIR,
)

tcav_multilayer.explain(cache_cav=True, cache_random=True)
tcav_multilayer.plot(figsize=(16, 5))

---

## 6. LocalVisualTCAV — Custom CAV function

By default, the CAV direction is computed as the difference between the mean
concept activation (positive centroid) and the mean random activation (negative centroid).

You can replace this with any custom function. The function must accept:
- `concept_features`: `torch.Tensor` of shape `[N, C]` — pooled concept activations
- `random_features`: `torch.Tensor` of shape `[N, C]` — pooled random activations

And return a `Cav` object with the `direction` field set.

### Example: normalized centroid difference

A simple variation that normalizes the direction to unit length.

In [ ]:
def normalized_centroid_cav(
    concept_features: torch.Tensor,
    random_features: torch.Tensor,
) -> Cav:
    """
    Custom CAV: centroid difference normalized to unit length.
    
    This ensures the direction vector has magnitude 1, which can
    improve comparability of attribution scores across different concepts.
    """
    positive_centroid = torch.mean(concept_features, dim=0)
    negative_centroid = torch.mean(random_features, dim=0)
    direction = positive_centroid - negative_centroid
    
    # Normalize to unit length
    direction = direction / (torch.norm(direction) + 1e-10)
    
    return Cav(direction=direction)


# Use the custom function by passing it as cav_fn
tcav_custom_cav = LocalVisualTCAV(
    model_wrapper=wrapper,
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    cache_dir=CACHE_DIR,
    cav_fn=normalized_centroid_cav,    # plug in your custom function here
)

tcav_custom_cav.explain(cache_cav=False)  # cache_cav=False to force recomputation
tcav_custom_cav.plot(figsize=(12, 5))

### Comparing default vs custom CAV

Let's compare the attribution scores from the default method vs our custom normalized method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get attribution scores from both methods
concepts = ["striped", "dotted"]
layer = "layer4"

# Top predicted class index
top_class = tcav_local.target_classes[0]

default_scores = [
    tcav_local.computations[layer][c].attributions.get(top_class, torch.tensor(0.0)).item()
    for c in concepts
]
custom_scores = [
    tcav_custom_cav.computations[layer][c].attributions.get(top_class, torch.tensor(0.0)).item()
    for c in concepts
]

x = np.arange(len(concepts))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, default_scores, width, label="Default (centroid diff)", color="steelblue")
ax.bar(x + width/2, custom_scores, width, label="Custom (normalized)", color="coral")

ax.set_xticks(x)
ax.set_xticklabels(concepts)
ax.set_ylabel("Attribution score")
ax.set_title(f"Default vs Custom CAV — layer4 — {wrapper.id_to_label(top_class)}")
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

---

## 7. GlobalVisualTCAV — Explaining a class across many images

`GlobalVisualTCAV` runs the pipeline on a folder of test images (e.g. 50 zebra photos)
and summarizes attribution scores statistically.

This answers: **"Does this concept consistently influence predictions for this class?"**

The result includes:
- **Mean attribution** — average influence of the concept across all images
- **Standard deviation** — how much it varies between images
- **95% confidence interval** — reliability of the estimate

In [ ]:
tcav_global = GlobalVisualTCAV(
    model_wrapper=wrapper,
    test_images_dir=os.path.join(TEST_IMAGES_DIR, "zebra"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    max_test_images=50,
    cache_dir=CACHE_DIR,
)

tcav_global.explain(cache_cav=True, cache_random=True)

### 7.1 Attribution statistics table

The table shows for each (concept, layer, class) combination:
- `Mean` — average attribution score across all test images
- `Std` — standard deviation
- `95% CI` — confidence interval [lower bound, upper bound]

A **high mean** with a **narrow confidence interval** means the concept
consistently and reliably influences the model's predictions.

In [ ]:
tcav_global.statsInfo()

### 7.2 Attribution bar chart

The bar chart shows mean attribution scores with error bars representing
the 95% confidence interval. Taller bars with smaller error bars indicate
more reliable concept influence.

In [ ]:
tcav_global.plot(figsize=(10, 5))

---

## 8. Saving results to disk

Both `plot()` methods accept a `save_path` parameter to save figures to disk
instead of displaying them. Useful for batch processing or paper figures.

In [ ]:
os.makedirs("./results", exist_ok=True)

# Save local explanation figure
tcav_local.plot(
    figsize=(14, 8),
    save_path="./results/zebra_local_explanation.png"
)

# Save global explanation figure
tcav_global.plot(
    figsize=(10, 5),
    save_path="./results/zebra_global_explanation.png"
)

print("Figures saved to ./results/")

---

## 9. Understanding the results

### Reading the concept map

- **Red/yellow areas** — the CNN strongly detects the concept here
- **Black areas** — the concept is absent
- The heatmap is the concept map upscaled from the layer's spatial resolution
  (7×7 for ResNet50 layer4) to the original image size (224×224)

### Reading the attribution score

- A score of **0.20** means the concept contributed approximately **20%** to the classification
- Higher scores for concepts that the model genuinely uses (e.g. `striped` for zebra)
- Near-zero scores for irrelevant concepts (e.g. `dotted` for zebra)

### Reading the global confidence interval

- **Narrow CI** (e.g. [0.18, 0.22]) — concept consistently matters across images ✅
- **Wide CI** (e.g. [0.02, 0.38]) — concept matters for some images but not others ⚠️
- **CI including 0** — concept does not reliably influence predictions ❌

---

## 10. Text-to-Concept (optional — requires CLIP)

The Text-to-Concept extension (by Daniele Di Santi) allows generating CAVs from
plain text descriptions instead of concept images.

Install CLIP first:
```bash
pip install visual-tcav[text-to-concept]
# or: pip install git+https://github.com/openai/CLIP.git
```

In [ ]:
try:
    from visual_tcav import TextToConcept

    # Load TextToConcept with a pre-trained Linear Aligner
    # (the aligner must be trained separately — see train_aligners.py)
    t2c = TextToConcept(model_wrapper=wrapper)
    t2c.load_aligner("./aligners/resnet50_layer4.pt")

    # Generate a CAV from just the word "stripes"
    cav_from_text = t2c.get_cav_from_text("stripes", layer_name="layer4")
    print(f"CAV generated from text: {cav_from_text}")
    print(f"Direction shape: {cav_from_text.direction.shape}")

except ImportError:
    print("CLIP not installed. Run: pip install visual-tcav[text-to-concept]")
except FileNotFoundError:
    print("Linear Aligner not found. Train it first with train_aligners.py")

---

## Summary

| Feature | How to use |
|---|---|
| **Minimal usage** | Pass all params to constructor, call `explain()`, call `plot()` |
| **Step-by-step** | Use `set_test_image()`, `set_concepts()`, `set_layers()` separately |
| **Skip predict()** | `explain()` calls it automatically if not done |
| **Multiple layers** | `layer_names=["layer2", "layer3", "layer4"]` |
| **Custom CAV** | Pass `cav_fn=my_function` to constructor |
| **Global explanation** | Use `GlobalVisualTCAV` with a folder of images |
| **Save figures** | `plot(save_path="./output.png")` |
| **Text-to-Concept** | `TextToConcept` + `load_aligner()` + `get_cav_from_text()` |
| **Caching** | `explain(cache_cav=True, cache_random=True)` (default) |

---

**GitHub:** https://github.com/saracavallini01/visual-tcav  
**PyPI:** `pip install visual-tcav`